# Tool Schemas

The function exists; now Claude needs to *know about it*. A **JSON schema** is the documentation Claude reads to decide **when** and **how** to call your tool. (JSON Schema is a general-purpose validation spec, not an AI thing — the ecosystem just adopted it as a convenient way to describe parameters.)

**Three parts of a tool spec:**

| Part | What it is |
|------|-----------|
| `name` | A clear, descriptive tool name (e.g. `get_current_datetime`) |
| `description` | What it does, *when* to use it, and what it returns |
| `input_schema` | JSON Schema for the arguments (types + per-arg descriptions) |

Still no Claude call here — we're only *defining* the schema. Wiring it into a request comes in **Handling message blocks**.

## The function (carried over)

In [1]:
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

## Its schema

Naming convention: `function_name` → **`function_name_schema`**, so each schema is easy to match to its function. We wrap it in **`ToolParam`** (from `anthropic.types`) — not required for functionality, but it gives type checking when you pass the schema to the API and catches typos in the schema shape.

In [2]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

get_current_datetime_schema

{'name': 'get_current_datetime',
 'description': 'Returns the current date and time formatted according to the specified format',
 'input_schema': {'type': 'object',
  'properties': {'date_format': {'type': 'string',
    'description': "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
    'default': '%Y-%m-%d %H:%M:%S'}},
  'required': []}}

## Writing effective descriptions

The `description` is the single biggest lever on whether Claude calls the tool correctly:

- Aim for **3–4 sentences** on what the tool does
- Say **when** Claude should use it
- Explain **what it returns**
- Give a **detailed description for each argument**

> The schema above is intentionally minimal to match the lesson. Compare it to the far richer `add_duration_to_datetime` / `set_reminder` descriptions coming in later lessons — those follow the 3–4 sentence guidance and read almost like mini docs.

## The easy way: let Claude write the schema

You rarely hand-write these. The workflow:
1. Copy your tool function's code
2. Ask Claude to write a JSON schema for tool calling for it
3. Include the Anthropic tool-use docs as context
4. Let Claude produce a best-practices-compliant schema

> Prompt: *"Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation."*